# Real-Time Facial Emotion Recognition
## ANN & CNN Deep Learning Mini Project

**Dataset:** Image_data (7 emotion classes)  
**Models:** Artificial Neural Network (ANN) baseline + Convolutional Neural Network (CNN)  
**Framework:** TensorFlow / Keras  
**Backend:** FastAPI  
**Frontend:** HTML, CSS, JavaScript


In [ ]:
# ─── Step 1: Environment & Library Setup ───
import os, sys, time, hashlib, pickle, warnings, random
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm
import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, regularizers
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, precision_score, recall_score, f1_score)
from sklearn.utils.class_weight import compute_class_weight

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

print(f"TensorFlow : {tf.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"OpenCV     : {cv2.__version__}")
print("Environment ready!")


In [ ]:
# ─── Configuration ───
BASE_DIR   = r"c:\Users\Anurag Rajendra\OneDrive\Pictures\PROJECTS\REAL TIME FACIAL EMOTION RECOGNITION"
DATA_DIR   = os.path.join(BASE_DIR, "Image_data")
MODELS_DIR = os.path.join(BASE_DIR, "models")
OUT_DIR    = os.path.join(BASE_DIR, "outputs")

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

EMOTION_CLASSES = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]
CLASS_TO_IDX    = {c: i for i, c in enumerate(EMOTION_CLASSES)}
IDX_TO_CLASS    = {i: c for c, i in CLASS_TO_IDX.items()}
NUM_CLASSES     = len(EMOTION_CLASSES)
IMG_SIZE        = (64, 64)
CHANNELS        = 1

print(f"Dataset    : {DATA_DIR}")
print(f"Classes    : {EMOTION_CLASSES}")
print(f"Input size : {IMG_SIZE}")


In [ ]:
# ─── Step 2: Dataset Inspection ───
print("Inspecting dataset...")
VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
records, hashes = [], {}
skipped = 0

for cls in EMOTION_CLASSES:
    cls_dir = os.path.join(DATA_DIR, cls)
    for fname in os.listdir(cls_dir):
        ext = os.path.splitext(fname)[1].lower()
        if ext not in VALID_EXTS:
            skipped += 1; continue
        fpath = os.path.join(cls_dir, fname)
        try:
            with Image.open(fpath) as img: img.verify()
        except Exception:
            skipped += 1; continue
        with open(fpath, "rb") as f:
            md5 = hashlib.md5(f.read()).hexdigest()
        if md5 in hashes:
            skipped += 1; continue
        hashes[md5] = fpath
        records.append((fpath, CLASS_TO_IDX[cls]))

print(f"Valid images : {len(records)}")
print(f"Skipped      : {skipped}")
class_counts = pd.Series([r[1] for r in records]).map(IDX_TO_CLASS).value_counts()
print("\nClass distribution:")
print(class_counts.to_string())


In [ ]:
# ─── Step 3-4: EDA – Class Distribution Plot ───
fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#e74c3c","#8e44ad","#3498db","#2ecc71","#95a5a6","#e67e22","#1abc9c"]
ax.bar(class_counts.index, class_counts.values, color=colors, alpha=0.9)
ax.set_title("Dataset Class Distribution", fontsize=16, fontweight="bold")
ax.set_xlabel("Emotion Class"); ax.set_ylabel("Number of Images")
for i, v in enumerate(class_counts.values):
    ax.text(i, v+80, str(v), ha="center", fontsize=9)
ax.set_facecolor("#f8f9fa"); fig.patch.set_facecolor("#f8f9fa")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "class_distribution.png"), dpi=150)
plt.show()
print("Plot saved to outputs/class_distribution.png")


In [ ]:
# ─── Step 5: Load & Preprocess Images ───
def load_image(path, size=IMG_SIZE, channels=CHANNELS):
    flag = cv2.IMREAD_GRAYSCALE if channels == 1 else cv2.IMREAD_COLOR
    img  = cv2.imread(path, flag)
    if img is None: return None
    img = cv2.resize(img, size)
    img = img.astype(np.float32) / 255.0
    if channels == 1: img = img[..., np.newaxis]
    return img

X_paths = [r[0] for r in records]
y_all   = np.array([r[1] for r in records])
X_all, valid_idx = [], []

for i, path in enumerate(tqdm(X_paths, desc="Loading")):
    img = load_image(path)
    if img is not None:
        X_all.append(img); valid_idx.append(i)

X_all = np.array(X_all, dtype=np.float32)
y_all = y_all[valid_idx]
print(f"X_all shape: {X_all.shape}")
print(f"y_all shape: {y_all.shape}")


In [ ]:
# ─── Step 6: Train/Test/Val Split (stratified 80:10:10) ───
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_all, y_all, test_size=0.20, random_state=SEED, stratify=y_all)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.125, random_state=SEED, stratify=y_train_full)

print(f"Train   : {X_train.shape[0]}")
print(f"Val     : {X_val.shape[0]}")
print(f"Test    : {X_test.shape[0]}")

# Class weights
class_weights_arr = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=y_train)
CLASS_WEIGHTS = {i: w for i, w in enumerate(class_weights_arr)}
print("\nClass weights:")
for i, w in CLASS_WEIGHTS.items(): print(f"  {IDX_TO_CLASS[i]}: {w:.3f}")


In [ ]:
# ─── Step 7: Sample Visualization ───
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()
plotted, idx = set(), 0
for img, lbl in zip(X_train, y_train):
    if lbl in plotted: continue
    axes[idx].imshow(img.squeeze(), cmap="gray")
    axes[idx].set_title(IDX_TO_CLASS[lbl].capitalize(), fontsize=12)
    axes[idx].axis("off")
    plotted.add(lbl); idx += 1
    if idx == NUM_CLASSES: break
axes[-1].axis("off")
plt.suptitle("Sample Images per Emotion Class", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "sample_images.png"), dpi=150)
plt.show()


In [ ]:
# ─── Step 8: ANN Architecture ───
X_train_flat = X_train.reshape(len(X_train), -1)
X_val_flat   = X_val.reshape(len(X_val), -1)
X_test_flat  = X_test.reshape(len(X_test), -1)

def build_ann(input_dim, num_classes):
    return keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(512, activation="relu", kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(), layers.Dropout(0.4),
        layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(), layers.Dropout(0.3),
        layers.Dense(128, activation="relu"), layers.Dropout(0.2),
        layers.Dense(num_classes, activation="softmax"),
    ], name="ANN_Emotion")

ann = build_ann(X_train_flat.shape[1], NUM_CLASSES)
ann.compile(optimizer=keras.optimizers.Adam(1e-3),
            loss="sparse_categorical_crossentropy", metrics=["accuracy"])
ann.summary()


In [ ]:
# ─── Step 9: Train ANN ───
ann_callbacks = [
    callbacks.EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6, verbose=1),
    callbacks.ModelCheckpoint(os.path.join(MODELS_DIR, "ann_model.keras"),
                              monitor="val_accuracy", save_best_only=True, verbose=0),
]
t0 = time.time()
ann_history = ann.fit(X_train_flat, y_train, validation_data=(X_val_flat, y_val),
    epochs=50, batch_size=64, class_weight=CLASS_WEIGHTS, callbacks=ann_callbacks, verbose=1)
ann_train_time = time.time() - t0
print(f"\nANN Training time: {ann_train_time:.1f}s")


In [ ]:
# ─── Step 10: Evaluate ANN ───
t1 = time.time()
y_pred_ann = np.argmax(ann.predict(X_test_flat, verbose=0), axis=1)
ann_inf_time = (time.time() - t1) / len(X_test)

ann_metrics = {
    "accuracy":   accuracy_score(y_test, y_pred_ann),
    "precision":  precision_score(y_test, y_pred_ann, average="weighted", zero_division=0),
    "recall":     recall_score(y_test, y_pred_ann, average="weighted", zero_division=0),
    "f1":         f1_score(y_test, y_pred_ann, average="weighted", zero_division=0),
    "train_time": ann_train_time,
    "inf_time":   ann_inf_time,
}
print(f"ANN Test Accuracy  : {ann_metrics['accuracy']:.4f}")
print(f"ANN Test Precision : {ann_metrics['precision']:.4f}")
print(f"ANN Test Recall    : {ann_metrics['recall']:.4f}")
print(f"ANN Test F1-Score  : {ann_metrics['f1']:.4f}")
print("\n" + classification_report(y_test, y_pred_ann, target_names=EMOTION_CLASSES))


In [ ]:
# ─── Step 11: ANN Training Curves & Confusion Matrix ───
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(ann_history.history["accuracy"], label="Train", color="#e74c3c", linewidth=2)
ax1.plot(ann_history.history["val_accuracy"], label="Val", color="#e74c3c", linewidth=2, linestyle="--")
ax1.set_title("ANN Accuracy"); ax1.set_xlabel("Epoch"); ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.plot(ann_history.history["loss"], label="Train", color="#e74c3c", linewidth=2)
ax2.plot(ann_history.history["val_loss"], label="Val", color="#e74c3c", linewidth=2, linestyle="--")
ax2.set_title("ANN Loss"); ax2.set_xlabel("Epoch"); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.suptitle("ANN Training History", fontsize=15, fontweight="bold")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "ann_history.png"), dpi=150); plt.show()

cm_ann = confusion_matrix(y_test, y_pred_ann)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm_ann, annot=True, fmt="d", cmap="Reds",
            xticklabels=EMOTION_CLASSES, yticklabels=EMOTION_CLASSES, ax=ax)
ax.set_title("ANN Confusion Matrix", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "ann_confusion_matrix.png"), dpi=150); plt.show()


In [ ]:
# ─── Step 12: CNN Architecture ───
INPUT_SHAPE_CNN = (*IMG_SIZE, CHANNELS)

data_aug = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.05, 0.05),
], name="data_augmentation")

def build_cnn(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)
    x = data_aug(inputs)
    for filters, pool in [(32, True), (64, True), (128, True), (256, False)]:
        x = layers.Conv2D(filters, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        if pool: x = layers.MaxPooling2D(2)(x)
        x = layers.Dropout(0.25)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="CNN_Emotion")

cnn = build_cnn(INPUT_SHAPE_CNN, NUM_CLASSES)
cnn.compile(optimizer=keras.optimizers.Adam(1e-3),
            loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cnn.summary()


In [ ]:
# ─── Step 13: Train CNN ───
cnn_callbacks = [
    callbacks.EarlyStopping(monitor="val_accuracy", patience=12, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1),
    callbacks.ModelCheckpoint(os.path.join(MODELS_DIR, "cnn_model.keras"),
                              monitor="val_accuracy", save_best_only=True, verbose=0),
]
t0 = time.time()
cnn_history = cnn.fit(X_train, y_train, validation_data=(X_val, y_val),
    epochs=60, batch_size=64, class_weight=CLASS_WEIGHTS, callbacks=cnn_callbacks, verbose=1)
cnn_train_time = time.time() - t0
print(f"\nCNN Training time: {cnn_train_time:.1f}s")


In [ ]:
# ─── Step 14: Evaluate CNN ───
t1 = time.time()
y_pred_cnn = np.argmax(cnn.predict(X_test, verbose=0), axis=1)
cnn_inf_time = (time.time() - t1) / len(X_test)

cnn_metrics = {
    "accuracy":   accuracy_score(y_test, y_pred_cnn),
    "precision":  precision_score(y_test, y_pred_cnn, average="weighted", zero_division=0),
    "recall":     recall_score(y_test, y_pred_cnn, average="weighted", zero_division=0),
    "f1":         f1_score(y_test, y_pred_cnn, average="weighted", zero_division=0),
    "train_time": cnn_train_time,
    "inf_time":   cnn_inf_time,
}
print(f"CNN Test Accuracy  : {cnn_metrics['accuracy']:.4f}")
print(f"CNN Test Precision : {cnn_metrics['precision']:.4f}")
print(f"CNN Test Recall    : {cnn_metrics['recall']:.4f}")
print(f"CNN Test F1-Score  : {cnn_metrics['f1']:.4f}")
print("\n" + classification_report(y_test, y_pred_cnn, target_names=EMOTION_CLASSES))


In [ ]:
# ─── Step 15: CNN Training Curves & Confusion Matrix ───
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(cnn_history.history["accuracy"], label="Train", color="#3498db", linewidth=2)
ax1.plot(cnn_history.history["val_accuracy"], label="Val", color="#3498db", linewidth=2, linestyle="--")
ax1.set_title("CNN Accuracy"); ax1.set_xlabel("Epoch"); ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.plot(cnn_history.history["loss"], label="Train", color="#3498db", linewidth=2)
ax2.plot(cnn_history.history["val_loss"], label="Val", color="#3498db", linewidth=2, linestyle="--")
ax2.set_title("CNN Loss"); ax2.set_xlabel("Epoch"); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.suptitle("CNN Training History", fontsize=15, fontweight="bold")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "cnn_history.png"), dpi=150); plt.show()

cm_cnn = confusion_matrix(y_test, y_pred_cnn)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm_cnn, annot=True, fmt="d", cmap="Blues",
            xticklabels=EMOTION_CLASSES, yticklabels=EMOTION_CLASSES, ax=ax)
ax.set_title("CNN Confusion Matrix", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "cnn_confusion_matrix.png"), dpi=150); plt.show()


In [ ]:
# ─── Step 16: ANN vs CNN Comparison ───
import numpy as np
metrics_names = ["Accuracy", "Precision", "Recall", "F1-Score"]
ann_vals = [ann_metrics["accuracy"], ann_metrics["precision"], ann_metrics["recall"], ann_metrics["f1"]]
cnn_vals = [cnn_metrics["accuracy"], cnn_metrics["precision"], cnn_metrics["recall"], cnn_metrics["f1"]]

x, w = np.arange(len(metrics_names)), 0.35
fig, ax = plt.subplots(figsize=(10, 6))
b1 = ax.bar(x-w/2, ann_vals, w, label="ANN", color="#e74c3c", alpha=0.85)
b2 = ax.bar(x+w/2, cnn_vals, w, label="CNN", color="#3498db", alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(metrics_names, fontsize=12)
ax.set_ylim(0, 1.1); ax.set_ylabel("Score")
ax.set_title("ANN vs CNN – Performance Comparison", fontsize=14, fontweight="bold")
ax.legend(fontsize=12); ax.grid(True, axis="y", alpha=0.3)
for bar in [*b1, *b2]:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f"{bar.get_height():.3f}", ha="center", fontsize=9)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "ann_vs_cnn_comparison.png"), dpi=150); plt.show()

# Summary Table
print("\n" + "="*52)
print(f"{'Metric':<22} {'ANN':>12} {'CNN':>12}")
print("-"*52)
for m, av, cv in [("Accuracy", ann_metrics["accuracy"], cnn_metrics["accuracy"]),
                   ("Precision", ann_metrics["precision"], cnn_metrics["precision"]),
                   ("Recall", ann_metrics["recall"], cnn_metrics["recall"]),
                   ("F1-Score", ann_metrics["f1"], cnn_metrics["f1"]),
                   ("Train Time (s)", ann_metrics["train_time"], cnn_metrics["train_time"]),
                   ("Inf Time (ms)", ann_metrics["inf_time"]*1000, cnn_metrics["inf_time"]*1000)]:
    print(f"{m:<22} {av:>12.4f} {cv:>12.4f}")
print("="*52)


In [ ]:
# ─── Step 17: Save All Artifacts ───
import json

ann.save(os.path.join(MODELS_DIR, "ann_model.keras"))
cnn.save(os.path.join(MODELS_DIR, "cnn_model.keras"))

preprocessing_cfg = {
    "img_size": IMG_SIZE, "channels": CHANNELS,
    "class_to_idx": CLASS_TO_IDX, "idx_to_class": IDX_TO_CLASS,
    "num_classes": NUM_CLASSES, "normalize": "divide_by_255",
    "color_mode": "grayscale"
}
ann_pkl = {"model_type":"ANN","model_path":os.path.join(MODELS_DIR,"ann_model.keras"),
           "metrics":{k:v for k,v in ann_metrics.items()},"img_size":IMG_SIZE,
           "channels":CHANNELS,"class_to_idx":CLASS_TO_IDX,"idx_to_class":IDX_TO_CLASS}
cnn_pkl = {"model_type":"CNN","model_path":os.path.join(MODELS_DIR,"cnn_model.keras"),
           "metrics":{k:v for k,v in cnn_metrics.items()},"img_size":IMG_SIZE,
           "channels":CHANNELS,"class_to_idx":CLASS_TO_IDX,"idx_to_class":IDX_TO_CLASS}

for name, obj in [("ann_model.pkl",ann_pkl),("cnn_model.pkl",cnn_pkl),
                   ("preprocessing_config.pkl",preprocessing_cfg)]:
    with open(os.path.join(MODELS_DIR, name), "wb") as f: pickle.dump(obj, f)

def make_serial(obj):
    if isinstance(obj, dict): return {k:make_serial(v) for k,v in obj.items()}
    if isinstance(obj, (list,tuple)): return [make_serial(v) for v in obj]
    if isinstance(obj, (float,)): return float(obj)
    return obj

ann_rpt = classification_report(y_test, y_pred_ann, target_names=EMOTION_CLASSES, output_dict=True)
cnn_rpt = classification_report(y_test, y_pred_cnn, target_names=EMOTION_CLASSES, output_dict=True)
with open(os.path.join(MODELS_DIR, "metrics.json"), "w") as f:
    json.dump(make_serial({"ann":{**ann_metrics,"report":ann_rpt},
                           "cnn":{**cnn_metrics,"report":cnn_rpt},
                           "ann_report":ann_rpt,"cnn_report":cnn_rpt}), f, indent=2)
with open(os.path.join(MODELS_DIR, "history.json"), "w") as f:
    json.dump({"ann":{k:[float(v) for v in vals] for k,vals in ann_history.history.items()},
               "cnn":{k:[float(v) for v in vals] for k,vals in cnn_history.history.items()}}, f, indent=2)

print("All artifacts saved!")
print("  models/ann_model.keras + ann_model.pkl")
print("  models/cnn_model.keras + cnn_model.pkl")
print("  models/preprocessing_config.pkl")
print("  models/metrics.json")
print("  models/history.json")


In [ ]:
# ─── Step 18: Sample Predictions Visualization ───
EMOTION_EMOJIS = {"angry":"😠","disgust":"🤢","fear":"😨","happy":"😊",
                   "neutral":"😐","sad":"😢","surprise":"😲"}

fig, axes = plt.subplots(2, 7, figsize=(18, 6))
sample_indices = [np.where(y_test==c)[0][0] for c in range(NUM_CLASSES)]

for col, idx in enumerate(sample_indices):
    img = X_test[idx]
    true_lbl = IDX_TO_CLASS[y_test[idx]]
    ann_p = ann.predict(img.reshape(1,-1), verbose=0)[0]
    cnn_p = cnn.predict(img[np.newaxis], verbose=0)[0]
    ann_pred, ann_conf = IDX_TO_CLASS[np.argmax(ann_p)], np.max(ann_p)
    cnn_pred, cnn_conf = IDX_TO_CLASS[np.argmax(cnn_p)], np.max(cnn_p)

    axes[0,col].imshow(img.squeeze(), cmap="gray")
    c = "green" if ann_pred==true_lbl else "red"
    axes[0,col].set_title(f"{EMOTION_EMOJIS.get(ann_pred,'')} {ann_pred}\n{ann_conf:.0%}", color=c, fontsize=9)
    axes[0,col].axis("off")

    axes[1,col].imshow(img.squeeze(), cmap="gray")
    c = "green" if cnn_pred==true_lbl else "red"
    axes[1,col].set_title(f"{EMOTION_EMOJIS.get(cnn_pred,'')} {cnn_pred}\n{cnn_conf:.0%}", color=c, fontsize=9)
    axes[1,col].axis("off")

axes[0,0].set_ylabel("ANN", fontsize=11)
axes[1,0].set_ylabel("CNN", fontsize=11)
plt.suptitle("Sample Predictions (Green=Correct, Red=Incorrect)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "sample_predictions.png"), dpi=150)
plt.show()


## Conclusion

The CNN significantly outperforms the ANN baseline for facial emotion recognition:

| Why CNN wins |
|---|
| **Spatial structure preserved** – Conv layers learn local patterns (edges, eyes, mouth) |
| **Hierarchical features** – Low→mid→high level facial features learned progressively |
| **Translation invariance** – MaxPooling makes predictions robust to small shifts |
| **Data augmentation** – Horizontal flip, rotation, zoom improve generalisation |

The ANN, while faster to train, loses spatial relationships by flattening the image,
resulting in lower accuracy—especially on visually similar emotions like Fear vs Surprise.

**To start the web application, run:**
```
python app.py
```
Then open http://localhost:8000
